# Mapa Interativo dos Hospitais Selecionados (Rio de Janeiro)

Este notebook cria um mapa interativo com os hospitais selecionados por `CNES`, usando latitude e longitude da base tratada.
O objetivo é facilitar a validação visual dos pontos para verificar se o local marcado corresponde a um hospital.

**O que será feito:**
- Carregar a base com coordenadas geográficas.
- Filtrar os hospitais selecionados.
- Plotar os pontos em um mapa com camadas de rua e satélite.
- Permitir inspeção individual de cada hospital com links para Google Maps, Street View e OpenStreetMap.


## 1. Dependências

Se estiver rodando este notebook em um ambiente novo, instale as bibliotecas abaixo uma única vez.


In [6]:
from pathlib import Path

import pandas as pd
import folium
from folium import plugins
from folium.plugins import MarkerCluster
from IPython.display import HTML, display

## 2. Carregamento da base de hospitais

A leitura prioriza o arquivo local do projeto


In [5]:
URL_HOSPITAIS = "https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/refs/heads/Refactoring-And-Documentation/Data/IntermediaryData/DataSus/respiratory_hospitalization_time_series_by_hospital_with_endereco.csv"

df_hospitais = pd.read_csv(URL_HOSPITAIS)

display(df_hospitais.head())
print(f"Total de linhas: {len(df_hospitais):,}".replace(",", "."))
print(f"Colunas disponíveis: {list(df_hospitais.columns)}")


,CNES,data_dia,num_internacoes,LAT,LON
0,2296748,2012-01-01,28,-22.752901,-43.406678
1,2296748,2012-01-02,98,-22.752901,-43.406678
2,2296748,2012-01-03,0,-22.752901,-43.406678
3,2296748,2012-01-04,14,-22.752901,-43.406678
4,2296748,2012-01-05,28,-22.752901,-43.406678


Total de linhas: 166.205
Colunas disponíveis: ['CNES', 'data_dia', 'num_internacoes', 'LAT', 'LON']


## 3. Filtragem e preparação dos pontos

Nesta etapa, filtramos apenas os hospitais de interesse e mantemos um ponto único por `CNES`.
Também validamos coordenadas para evitar marcadores inválidos no mapa.


In [7]:
def normalizar_cnes(valor):
    texto = str(valor).strip()
    if texto.endswith(".0"):
        texto = texto[:-2]
    return texto


SELECTED_HOSP = [
    "2296748", "2269384", "2295423", "2291266", "2269481", "2798662", "2277751",
    "2270269", "2296306", "2269341", "2269724", "2270609", "2269783", "2296616",
    "2280167", "2273411", "2270234", "2298120",
]


df_hospitais = df_hospitais.copy()
df_hospitais["CNES"] = df_hospitais["CNES"].map(normalizar_cnes)
df_hospitais["LAT"] = pd.to_numeric(df_hospitais["LAT"], errors="coerce")
df_hospitais["LON"] = pd.to_numeric(df_hospitais["LON"], errors="coerce")

df_filtrado = df_hospitais[df_hospitais["CNES"].isin(SELECTED_HOSP)].copy()

df_unique_hospitais = (
    df_filtrado[["CNES", "LAT", "LON"]]
    .dropna(subset=["LAT", "LON"])
    .drop_duplicates(subset=["CNES", "LAT", "LON"])
    .sort_values("CNES")
    .reset_index(drop=True)
)

cnes_encontrados = set(df_unique_hospitais["CNES"])
cnes_nao_encontrados = sorted(set(SELECTED_HOSP) - cnes_encontrados)

print(f"Hospitais selecionados: {len(SELECTED_HOSP)}")
print(f"Hospitais com coordenadas válidas: {df_unique_hospitais['CNES'].nunique()}")

if cnes_nao_encontrados:
    print("CNES sem coordenada válida na base:")
    print(cnes_nao_encontrados)

display(df_unique_hospitais.head())

Hospitais selecionados: 18
Hospitais com coordenadas válidas: 18


,CNES,LAT,LON
0,2269341,-22.911859,-43.243431
1,2269384,-22.927659,-43.252612
2,2269481,-22.891583,-43.309908
3,2269724,-22.814954,-43.225515
4,2269783,-22.914309,-43.238253


## 4. Mapa interativo dos hospitais

O mapa abaixo permite inspeção visual com zoom, troca de camada e clique nos marcadores.
Cada popup contém links para validação rápida da localização.


In [8]:
def gerar_links_validacao(lat, lon):
    return {
        "google_maps": f"https://www.google.com/maps?q={lat},{lon}",
        "street_view": f"https://www.google.com/maps/@?api=1&map_action=pano&viewpoint={lat},{lon}",
        "openstreetmap": f"https://www.openstreetmap.org/?mlat={lat}&mlon={lon}#map=18/{lat}/{lon}",
    }


def criar_popup_hospital(cnes, lat, lon):
    links = gerar_links_validacao(lat, lon)
    html = (
        "<div style='font-size:13px;'>"
        f"<b>CNES:</b> {cnes}<br>"
        f"<b>Latitude:</b> {lat:.6f}<br>"
        f"<b>Longitude:</b> {lon:.6f}<br><br>"
        f"<a href='{links['google_maps']}' target='_blank'>Abrir no Google Maps</a><br>"
        f"<a href='{links['street_view']}' target='_blank'>Abrir no Street View</a><br>"
        f"<a href='{links['openstreetmap']}' target='_blank'>Abrir no OpenStreetMap</a>"
        "</div>"
    )
    return html


def criar_mapa_hospitais(df_pontos, cnes_destaque=None, zoom_inicial=11):
    if df_pontos.empty:
        raise ValueError("O DataFrame está vazio. Não há hospitais para plotar.")

    centro = [df_pontos["LAT"].mean(), df_pontos["LON"].mean()]
    mapa = folium.Map(location=centro, zoom_start=zoom_inicial, control_scale=True, tiles=None)

    folium.TileLayer("OpenStreetMap", name="Mapa de ruas", show=True).add_to(mapa)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Tiles (c) Esri",
        name="Satélite (Esri)",
        show=False,
    ).add_to(mapa)

    cluster = MarkerCluster(name="Hospitais").add_to(mapa)

    cnes_destaque = str(cnes_destaque) if cnes_destaque is not None else None

    for row in df_pontos.itertuples(index=False):
        cnes = str(row.CNES)
        lat = float(row.LAT)
        lon = float(row.LON)
        destaque = cnes == cnes_destaque

        marker = folium.Marker(
            location=[lat, lon],
            tooltip=f"CNES {cnes}",
            popup=folium.Popup(criar_popup_hospital(cnes, lat, lon), max_width=360),
            icon=folium.Icon(color="red" if destaque else "blue", icon="plus-sign"),
        )
        marker.add_to(cluster)

        if destaque:
            folium.Circle(
                location=[lat, lon],
                radius=180,
                color="red",
                fill=True,
                fill_opacity=0.12,
                weight=2,
            ).add_to(mapa)
            mapa.location = [lat, lon]
            mapa.zoom_start = 16

    plugins.Fullscreen(position="topleft").add_to(mapa)
    plugins.MeasureControl(position="topleft", primary_length_unit="meters").add_to(mapa)
    plugins.MousePosition(position="bottomright").add_to(mapa)
    folium.LayerControl(collapsed=False).add_to(mapa)

    return mapa

In [9]:
mapa_hospitais = criar_mapa_hospitais(df_unique_hospitais)
mapa_hospitais